# FASE 2: MODEL SELECTION EXPERIMENT (GOOGLE COLAB GPU T4)
## Deteksi Penyakit, Hama, dan Daun Sehat Tanaman Teh (Resolusi 640x640)

Notebook ini dirancang untuk menjalankan eksperimen perbandingan baseline object detection:
1. **Faster R-CNN (ResNet-50-FPN)**
2. **RetinaNet (ResNet-50-FPN)**
3. **SSD (SSD300-VGG16)**
4. **RT-DETR (RT-DETR-L)**
5. **YOLO26n Baseline**

> **Keuntungan GPU T4 di Google Colab:**
> - Resolusi tetap **640x640** (sesuai instruksi dosen pembimbing)
> - 50 Epoch selesai hanya dalam **~30–35 Menit** (dibandingkan 20–50 jam di CPU lokal)
> - Laporan lengkap (`laporan_hasil_run.md`) otomatis terbuat di direktori masing-masing model.

### Langkah 1: Pastikan Akselerator GPU Aktif
Klik menu **Runtime** di atas $\to$ **Change runtime type** $	o$ Pilih **T4 GPU** $	o$ Klik **Save**.

In [ ]:
!nvidia-smi

### Langkah 2: Hubungkan ke Google Drive
Hubungkan Google Drive yang berisi file `tea_yolo.zip` dan `model_selection.zip` di dalam folder `SKRIPSI`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Langkah 3: Ekstrak Dataset & Kode ke Disk Lokal Colab (NVMe SSD)
Mengekstrak ke `/content/` membuat proses membaca gambar menjadi 10x lebih cepat dibanding membaca langsung dari Drive.

In [ ]:
# Salin berkas zip dari Google Drive ke disk lokal Colab
!cp /content/drive/MyDrive/SKRIPSI/tea_yolo.zip /content/
!cp /content/drive/MyDrive/SKRIPSI/model_selection.zip /content/ 2>/dev/null || true

# Ekstrak berkas dataset dan kode
!unzip -q /content/tea_yolo.zip -d /content/
!unzip -q /content/model_selection.zip -d /content/ 2>/dev/null || true

# Pastikan yolo26n.pt tersedia di /content/
!cp /content/model_selection/yolo26n.pt /content/ 2>/dev/null || true

# Sinkronisasi otomatis data.yaml untuk Colab
!python -m model_selection.config

print("Ekstraksi dan konfigurasi dataset selesai!")
!cat /content/tea_yolo/data.yaml


### Langkah 4: Install Dependencies Pendukung

In [ ]:
!pip install -q ultralytics faster-coco-eval thop

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device     :", torch.cuda.get_device_name(0))
    print("VRAM Terpasang :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

---
## EKSEKUSI MODEL 1 PER TAHAP (STEP-BY-STEP)
Anda dapat menjalankan cell di bawah ini satu per satu sesuai model yang ingin Anda uji!

### Model 1: Faster R-CNN (ResNet-50-FPN) — Resolusi 640x640

In [ ]:
# 1. Pelatihan Faster R-CNN (50 Epochs, batch 8, resolusi 640x640, GPU CUDA)
!python -m model_selection.train_detector --model faster_rcnn --epochs 50 --batch_size 8 --imgsz 640 --device cuda

# 2. Evaluasi Test Set (529 gambar)
!python -m model_selection.evaluate_detector --model faster_rcnn --imgsz 640 --device cuda

# 3. Analisis Visual Kesalahan (TP, FP, FN, Misklasifikasi)
!python -m model_selection.visualize_predictions --model faster_rcnn --imgsz 640 --device cuda

# 4. Salin Hasil & Laporan ke Google Drive agar tersimpan permanen
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase2_runs/faster_rcnn/
!cp -r /content/phase2_runs/model_selection/faster_rcnn/* /content/drive/MyDrive/SKRIPSI/phase2_runs/faster_rcnn/
print("\n[SELESAI] Hasil Faster R-CNN dan laporan_hasil_run.md telah tersimpan permanen di Google Drive!")

### Model 2: RetinaNet (ResNet-50-FPN) — Resolusi 640x640

In [ ]:
# 1. Pelatihan RetinaNet (50 Epochs, batch 8, resolusi 640x640, GPU CUDA)
!python -m model_selection.train_detector --model retinanet --epochs 50 --batch_size 8 --imgsz 640 --device cuda

# 2. Evaluasi Test Set
!python -m model_selection.evaluate_detector --model retinanet --imgsz 640 --device cuda

# 3. Analisis Visual Kesalahan
!python -m model_selection.visualize_predictions --model retinanet --imgsz 640 --device cuda

# 4. Salin Hasil & Laporan ke Google Drive
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase2_runs/retinanet/
!cp -r /content/phase2_runs/model_selection/retinanet/* /content/drive/MyDrive/SKRIPSI/phase2_runs/retinanet/
print("\n[SELESAI] Hasil RetinaNet telah tersimpan di Google Drive!")

### Model 3: SSD (SSD300-VGG16)

In [ ]:
!python -m model_selection.train_detector --model ssd --epochs 50 --batch_size 8 --imgsz 640 --device cuda
!python -m model_selection.evaluate_detector --model ssd --imgsz 640 --device cuda
!python -m model_selection.visualize_predictions --model ssd --imgsz 640 --device cuda

!mkdir -p /content/drive/MyDrive/SKRIPSI/phase2_runs/ssd/
!cp -r /content/phase2_runs/model_selection/ssd/* /content/drive/MyDrive/SKRIPSI/phase2_runs/ssd/
print("\n[SELESAI] Hasil SSD telah tersimpan di Google Drive!")

### Model 4: RT-DETR (RT-DETR-L Transformer)

In [ ]:
!python -m model_selection.train_detector --model rtdetr --epochs 50 --batch_size 8 --imgsz 640 --device cuda
!python -m model_selection.evaluate_detector --model rtdetr --imgsz 640 --device cuda
!python -m model_selection.visualize_predictions --model rtdetr --imgsz 640 --device cuda

!mkdir -p /content/drive/MyDrive/SKRIPSI/phase2_runs/rtdetr/
!cp -r /content/phase2_runs/model_selection/rtdetr/* /content/drive/MyDrive/SKRIPSI/phase2_runs/rtdetr/
print("\n[SELESAI] Hasil RT-DETR telah tersimpan di Google Drive!")

### Model 5: YOLO26n Baseline (Anchor-Free CNN) — Resolusi 640x640
Model pembanding baseline resmi untuk melihat performa detektor Anchor-Free CNN pada resolusi identik 640x640.

In [ ]:
# 1. Pelatihan YOLO26n Baseline (50 Epochs, batch 8, resolusi 640x640, GPU CUDA)
!python -m model_selection.train_detector --model yolo26n --epochs 50 --batch_size 8 --imgsz 640 --device cuda

# 2. Evaluasi Test Set (529 gambar)
!python -m model_selection.evaluate_detector --model yolo26n --imgsz 640 --device cuda

# 3. Analisis Visual Kesalahan (TP, FP, FN, Misklasifikasi)
!python -m model_selection.visualize_predictions --model yolo26n --imgsz 640 --device cuda

# 4. Salin Hasil & Laporan ke Google Drive agar tersimpan permanen
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase2_runs/yolo26n/
!cp -r /content/phase2_runs/model_selection/yolo26n/* /content/drive/MyDrive/SKRIPSI/phase2_runs/yolo26n/
print("\n[SELESAI] Hasil YOLO26n telah tersimpan di Google Drive!")

---
## LIHAT LAPORAN HASIL RUN LANGSUNG DI COLAB
Jalankan cell di bawah untuk menampilkan isi laporan hasil evaluasi model yang baru saja selesai.

In [ ]:
# Ganti 'faster_rcnn' dengan nama model yang ingin dilihat laporannya
model_laporan = "faster_rcnn"
report_path = f"/content/phase2_runs/model_selection/{model_laporan}/laporan_hasil_run.md"

from IPython.display import display, Markdown
import os

if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print(f"Laporan {report_path} belum tersedia. Jalankan training terlebih dahulu.")